# Cassava Leaf Disease Classification

| Label | Disease |
|-------|----------|
| 0 | Cassava Bacterial Blight (CBB) |
| 1 | Cassava Brown Streak Disease (CBSD) |
| 2 | Cassava Green Mottle (CGM) |
| 3 | Cassava Mosaic Disease (CMD) |
| 4 | Healthy |

## Summary

### Changes vs V1

| Component | V1 | V2 (This Notebook) |
|-----------|----|-----------------|
| **Model** | `efficientnet_b4` | `tf_efficientnet_b4_ns` (NoisyStudent pretrain) |
| **Image Size** | 380×380 | 512×512 |
| **Learning Rate** | `1e-4` | `3e-4` |
| **Scheduler** | `CosineAnnealingLR` (T_max=10) | `CosineAnnealingWarmRestarts` (T_0=5×steps) |
| **Epochs** | 10 | 12 |
| **Mixed Precision** | No | Yes (AMP) |
| **Augmentation** | Resize, HFlip, VFlip, Rotate(30°), ColorJitter(0.2), GaussNoise/Blur | RandomResizedCrop, HFlip, VFlip, Rotate(45°), ColorJitter(0.3), GaussNoise/Blur/MotionBlur |
| **Loss** | CrossEntropyLoss (label_smoothing=0.1) | CrossEntropyLoss (label_smoothing=0.1) — unchanged |
| **CV Training** | Fold 0 only | Fold 0 only |

### Motivation

- **NoisyStudent (`tf_efficientnet_b4_ns`)**: pretrained with noisy student self-training on ImageNet. Significantly more robust on real-world noisy/imbalanced datasets like cassava.
- **Larger resolution (512)**: captures finer disease texture details that 380px misses.
- **AMP**: halves GPU memory usage and speeds up training with no accuracy cost.
- **Warm restarts**: periodic LR resets help escape local minima; combined with a higher base LR of 3e-4, convergence is faster and more stable.
- **Stronger augmentation**: `RandomResizedCrop` forces the model to be scale-invariant; `MotionBlur` simulates camera shake common in field photos; tighter ColorJitter and wider rotation improve domain robustness.

### Key Hyperparameters

- **Model**: `tf_efficientnet_b4_ns` (NoisyStudent pretrained)
- **Image size**: 512 × 512
- **Batch size**: 16
- **Epochs**: 12
- **Optimizer**: AdamW
- **Learning rate**: 3e-4
- **Weight decay**: 1e-5
- **Scheduler**: CosineAnnealingWarmRestarts (T_0=5×steps_per_epoch, eta_min=1e-6)
- **Loss**: CrossEntropyLoss (label_smoothing=0.1)
- **CV setup**: StratifiedKFold (5 folds; trained on fold 0)

In [ ]:
# Install / upgrade dependencies
!pip install -q timm "albumentations==1.3.1" opencv-python-headless

In [ ]:
import os
import json
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# AMP — prefer modern torch.amp, fall back for older torch
try:
    from torch.amp import GradScaler, autocast
    _AMP_DEVICE_ARG = True
except ImportError:
    from torch.cuda.amp import GradScaler, autocast
    _AMP_DEVICE_ARG = False

from contextlib import nullcontext

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedKFold

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
class CFG:
    # Paths (Kaggle)
    data_dir      = '/kaggle/input/cassava-leaf-disease-classification'
    train_csv     = data_dir + '/train.csv'
    train_images  = data_dir + '/train_images'
    test_images   = data_dir + '/test_images'
    label_map_path = data_dir + '/label_num_to_disease_map.json'
    sample_submission = data_dir + '/sample_submission.csv'
    output_dir    = '/kaggle/working'

    # Model
    model_name    = 'tf_efficientnet_b4_ns'   # NoisyStudent — better on noisy labels
    pretrained    = True
    num_classes   = 5
    model_save_prefix = 'best_model_v2'

    # Training
    seed          = 42
    num_folds     = 5
    fold          = 0
    num_epochs    = 12
    batch_size    = 16
    num_workers   = 2
    image_size    = 512

    # Optimizer
    lr            = 3e-4
    weight_decay  = 1e-5

    # Scheduler
    T_0           = 5          # warm-restart period in epochs
    eta_min       = 1e-6

    # Loss
    label_smoothing = 0.1

    # AMP
    use_amp       = torch.cuda.is_available()

    # ImageNet stats
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CFG.seed)

assert os.path.exists(CFG.data_dir),   f'data_dir not found: {CFG.data_dir}'
assert os.path.exists(CFG.train_csv),  f'train.csv not found: {CFG.train_csv}'
print('Config loaded.')
print(f'  model     : {CFG.model_name}')
print(f'  image_size: {CFG.image_size}')
print(f'  use_amp   : {CFG.use_amp}')

In [ ]:
# Load training CSV and label map
train_df = pd.read_csv(CFG.train_csv)
print(f'Train samples: {len(train_df)}')
print(train_df.head())

with open(CFG.label_map_path, 'r') as f:
    label_map = {int(k): v for k, v in json.load(f).items()}

print('\nLabel map:')
for k, v in label_map.items():
    print(f'  {k}: {v}')

class_counts = train_df['label'].value_counts().sort_index()
class_names  = [label_map[i] for i in class_counts.index]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(class_names, class_counts.values,
              color=['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db'])
ax.set_title('Class Distribution in Training Set', fontsize=14, fontweight='bold')
ax.set_xlabel('Disease Class')
ax.set_ylabel('Number of Images')
ax.tick_params(axis='x', rotation=20)
for bar, count in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
            str(count), ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

print('\nClass counts:')
for i, (name, count) in enumerate(zip(class_names, class_counts.values)):
    print(f'  [{i}] {name}: {count} ({100*count/len(train_df):.1f}%)')

## Exploratory Data Analysis

Visualise sample images from each class to understand visual characteristics and guide augmentation choices.

In [ ]:
n_samples = 4
fig, axes = plt.subplots(CFG.num_classes, n_samples,
                         figsize=(4 * n_samples, 4 * CFG.num_classes))

for cls_idx in range(CFG.num_classes):
    cls_df = train_df[train_df['label'] == cls_idx].sample(n_samples, random_state=CFG.seed)
    for col, (_, row) in enumerate(cls_df.iterrows()):
        img_path = os.path.join(CFG.train_images, row['image_id'])
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[cls_idx, col].imshow(img)
        axes[cls_idx, col].axis('off')
        if col == 0:
            axes[cls_idx, col].set_ylabel(
                f'[{cls_idx}] {label_map[cls_idx]}',
                fontsize=10, fontweight='bold', rotation=0,
                labelpad=120, va='center'
            )

fig.suptitle('Sample Images per Disease Class', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Augmentation transforms
# Key changes vs V1:
#   - RandomResizedCrop (scale=0.7-1.0) replaces plain Resize — forces scale invariance
#   - Rotate limit increased from 30 to 45 degrees
#   - ColorJitter strength increased from 0.2 to 0.3
#   - MotionBlur added to noise/blur OneOf block

def get_train_transforms():
    return A.Compose([
        A.RandomResizedCrop(CFG.image_size, CFG.image_size, scale=(0.7, 1.0), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=45, p=0.5),
        A.ColorJitter(
            brightness=0.3, contrast=0.3,
            saturation=0.3, hue=0.1, p=0.5
        ),
        A.OneOf([
            A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),
            A.GaussianBlur(blur_limit=(3, 7), p=1.0),
            A.MotionBlur(blur_limit=7, p=1.0),
        ], p=0.3),
        A.Normalize(mean=CFG.mean, std=CFG.std),
        ToTensorV2(),
    ])

def get_val_transforms():
    return A.Compose([
        A.Resize(CFG.image_size, CFG.image_size),
        A.Normalize(mean=CFG.mean, std=CFG.std),
        ToTensorV2(),
    ])

def get_test_transforms():
    return A.Compose([
        A.Resize(CFG.image_size, CFG.image_size),
        A.Normalize(mean=CFG.mean, std=CFG.std),
        ToTensorV2(),
    ])

print('Transforms defined.')
print('  Train: RandomResizedCrop, HFlip, VFlip, Rotate(45°), ColorJitter(0.3), GaussNoise/Blur/MotionBlur')
print('  Val/Test: Resize + Normalize only')

In [ ]:
class CassavaDataset(Dataset):
    """Custom PyTorch Dataset for Cassava Leaf Disease images."""

    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['image_id'])

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            image = self.transform(image=image)['image']

        if self.is_test:
            return image

        label = torch.tensor(row['label'], dtype=torch.long)
        return image, label


print('CassavaDataset defined.')

In [ ]:
# StratifiedKFold split
skf = StratifiedKFold(n_splits=CFG.num_folds, shuffle=True, random_state=CFG.seed)

train_df['fold'] = -1
for fold_idx, (_, val_idx) in enumerate(skf.split(train_df, train_df['label'])):
    train_df.loc[val_idx, 'fold'] = fold_idx

print('Fold distribution:')
for f in range(CFG.num_folds):
    n = (train_df['fold'] == f).sum()
    print(f'  Fold {f}: {n} samples')

fold_train_df = train_df[train_df['fold'] != CFG.fold].reset_index(drop=True)
fold_val_df   = train_df[train_df['fold'] == CFG.fold].reset_index(drop=True)
print(f'\nUsing fold {CFG.fold}:')
print(f'  Train: {len(fold_train_df)} | Val: {len(fold_val_df)}')

train_dataset = CassavaDataset(fold_train_df, CFG.train_images, transform=get_train_transforms())
val_dataset   = CassavaDataset(fold_val_df,   CFG.train_images, transform=get_val_transforms())

train_loader = DataLoader(
    train_dataset, batch_size=CFG.batch_size, shuffle=True,
    num_workers=CFG.num_workers, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=CFG.batch_size, shuffle=False,
    num_workers=CFG.num_workers, pin_memory=True,
)

print(f'  Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## Model Architecture

We upgrade from **EfficientNet-B4** to **`tf_efficientnet_b4_ns`** — the NoisyStudent variant.

NoisyStudent pretraining uses a teacher-student loop where the student is trained on teacher pseudo-labels with noise injection (dropout, stochastic depth, data augmentation). This produces representations that are significantly more robust to label noise, which is a key challenge in the cassava dataset where inter-class visual similarity is high.

In [ ]:
def build_model(pretrained=CFG.pretrained):
    model = timm.create_model(CFG.model_name, pretrained=pretrained)
    n_features = model.classifier.in_features
    model.classifier = nn.Linear(n_features, CFG.num_classes)
    return model.to(device)


model = build_model()

total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {CFG.model_name}')
print(f'  Total parameters    : {total_params:,}')
print(f'  Trainable parameters: {trainable_params:,}')

In [ ]:
# Loss function — unchanged from V1
criterion = nn.CrossEntropyLoss(label_smoothing=CFG.label_smoothing)
print(f'Loss: CrossEntropyLoss(label_smoothing={CFG.label_smoothing})')

# AMP scaler
scaler = GradScaler() if CFG.use_amp else None
print(f'AMP : {CFG.use_amp} (scaler={scaler is not None})')


def _autocast_ctx():
    """Return the correct autocast context for this torch version."""
    if not CFG.use_amp or device.type != 'cuda':
        return nullcontext()
    if _AMP_DEVICE_ARG:
        return autocast(device_type='cuda')
    return autocast()

In [ ]:
# Optimizer + Scheduler
# CosineAnnealingWarmRestarts steps per-batch (not per-epoch),
# so T_0 is expressed in number of batches = CFG.T_0 * steps_per_epoch.
optimizer = optim.AdamW(
    model.parameters(),
    lr=CFG.lr,
    weight_decay=CFG.weight_decay
)

scheduler = CosineAnnealingWarmRestarts(
    optimizer,
    T_0=CFG.T_0 * len(train_loader),
    eta_min=CFG.eta_min
)

print(f'Optimizer : AdamW (lr={CFG.lr}, weight_decay={CFG.weight_decay})')
print(f'Scheduler : CosineAnnealingWarmRestarts')
print(f'            T_0={CFG.T_0}×{len(train_loader)} batches = {CFG.T_0*len(train_loader)} steps')
print(f'            eta_min={CFG.eta_min}')

## Training

Key changes vs V1:
- `scheduler.step()` is called **inside** the batch loop (per-step warm restarts instead of per-epoch)
- Forward/backward passes are wrapped in **AMP** `autocast` + `GradScaler`
- `optimizer.zero_grad(set_to_none=True)` reduces memory by deallocating grad tensors instead of zeroing them

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, scheduler, device, epoch):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc=f'Epoch {epoch+1} [Train]', leave=False)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with _autocast_ctx():
            outputs = model(images)
            loss = criterion(outputs, labels)

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        # Step scheduler per batch for warm restarts
        scheduler.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{correct/total:.4f}'})

    return running_loss / total, correct / total


print('train_one_epoch() defined.')

In [ ]:
def validate_one_epoch(model, loader, criterion, device, epoch):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc=f'Epoch {epoch+1} [Val]  ', leave=False)
    with torch.no_grad():
        for images, labels in pbar:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with _autocast_ctx():
                outputs = model(images)
                loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{correct/total:.4f}'})

    return running_loss / total, correct / total


print('validate_one_epoch() defined.')

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

best_val_acc = 0.0
best_model_path = os.path.join(CFG.output_dir, f'{CFG.model_save_prefix}_fold{CFG.fold}.pth')

print(f'Starting training for {CFG.num_epochs} epochs...')
print(f'{"Epoch":>5} | {"Train Loss":>10} | {"Train Acc":>9} | {"Val Loss":>8} | {"Val Acc":>7} | {"LR":>10}')
print('-' * 65)

for epoch in range(CFG.num_epochs):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, scheduler, device, epoch
    )
    val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device, epoch)

    current_lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f'{epoch+1:>5} | {train_loss:>10.4f} | {train_acc:>9.4f} | {val_loss:>8.4f} | {val_acc:>7.4f} | {current_lr:>10.2e}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f'         --> New best model saved! Val Acc: {best_val_acc:.4f}')

print(f'\nTraining complete. Best Val Acc: {best_val_acc:.4f}')

In [ ]:
# Plot training curves
epochs_range = range(1, CFG.num_epochs + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss', linewidth=2)
ax1.plot(epochs_range, history['val_loss'],   'r-o', label='Val Loss',   linewidth=2)
ax1.set_title('Training & Validation Loss', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history['train_acc'], 'b-o', label='Train Acc', linewidth=2)
ax2.plot(epochs_range, history['val_acc'],   'r-o', label='Val Acc',   linewidth=2)
ax2.axhline(y=best_val_acc, color='green', linestyle='--', alpha=0.7,
            label=f'Best Val Acc: {best_val_acc:.4f}')
ax2.set_title('Training & Validation Accuracy', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle(f'{CFG.model_name} — Fold {CFG.fold} Training Curves',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## Inference

Load the best checkpoint and run inference on the test set. No TTA at this stage — that will be added in V3.

In [ ]:
inference_model = build_model(pretrained=False)
inference_model.load_state_dict(torch.load(best_model_path, map_location=device))
inference_model.eval()

print(f'Loaded best model from: {best_model_path}')
print(f'Best validation accuracy: {best_val_acc:.4f}')

In [ ]:
test_df = pd.read_csv(CFG.sample_submission)
print(f'Test samples: {len(test_df)}')
print(test_df.head())

test_dataset = CassavaDataset(
    test_df, CFG.test_images,
    transform=get_test_transforms(),
    is_test=True
)
test_loader = DataLoader(
    test_dataset, batch_size=CFG.batch_size, shuffle=False,
    num_workers=CFG.num_workers, pin_memory=True,
)

all_preds = []
with torch.no_grad():
    for images in tqdm(test_loader, desc='Inference'):
        images = images.to(device, non_blocking=True)
        with _autocast_ctx():
            outputs = inference_model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)

print(f'\nInference complete. Predictions: {len(all_preds)}')
unique, counts = np.unique(all_preds, return_counts=True)
print('Prediction distribution:')
for cls, cnt in zip(unique, counts):
    print(f'  [{cls}] {label_map[cls]}: {cnt}')

In [ ]:
submission_df = pd.DataFrame({'image_id': test_df['image_id'], 'label': all_preds})
submission_path = os.path.join(CFG.output_dir, 'submission.csv')
submission_df.to_csv(submission_path, index=False)

print(f'Submission saved to: {submission_path}')
print(f'Shape: {submission_df.shape}')
print(submission_df.head(10))

assert len(submission_df) == len(test_df), 'Row count mismatch!'
assert submission_df['label'].between(0, 4).all(), 'Invalid label values!'
print('\nSanity checks passed. Ready to submit!')